# ASR Shootout — Exploration Notebook

Interactive analysis of benchmark results. Run `bash run.sh` first to generate metrics.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RESULTS = ROOT / "data" / "outputs" / "metrics" / "full_results.csv"

In [ ]:
df = pd.read_csv(RESULTS)
print(f"Loaded {len(df)} rows, {df['model_name'].nunique()} models, {df['filename'].nunique()} files")
df.head()

## Leaderboard — Entity Accuracy vs WER

In [ ]:
summary = df.groupby("model_name").agg(
    mean_wer=("wer", "mean"),
    entity_acc=("locality_correct", "mean"),
    latency=("latency_seconds", "mean"),
).round(4)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for model, row in summary.iterrows():
    ax.scatter(row["mean_wer"], row["entity_acc"], s=120, label=model)
    ax.annotate(model, (row["mean_wer"], row["entity_acc"]), fontsize=9)
ax.set_xlabel("Mean WER")
ax.set_ylabel("Locality Entity Accuracy")
ax.set_title("WER vs Entity Accuracy (upper-left is better)")
plt.tight_layout()
plt.show()

## Noise Condition Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df, x="condition", y="wer", hue="model_name", ax=ax)
ax.set_title("WER by Recording Condition")
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## Failure Samples — Dropped Localities

In [ ]:
failures = df[~df["locality_correct"]].sort_values("wer", ascending=False)
failures[["filename", "model_name", "locality_name", "wer", "reference_transcript", "hypothesis"]].head(15)

## Single-File Transcription Test

In [ ]:
from src.models.faster_whisper_asr import FasterWhisperASR
from src.models.registry import get_active_models

audio_files = list((ROOT / "data" / "raw").rglob("*.wav"))
if audio_files:
    model = FasterWhisperASR()
    result = model.transcribe(audio_files[0])
    print(f"File: {audio_files[0].name}")
    print(f"Transcript: {result.transcript}")
    print(f"Latency: {result.latency_seconds:.2f}s")
else:
    print("No audio files found. Run: python scripts/prepare_demo_data.py")